In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import anthropic
import google.generativeai
from IPython.display import display, Markdown, update_display

In [2]:
# Config
system_message = "You are an assistant that is great in telling jokes."
user_prompt = "Tell a light-hearted joke for and audience of Data Scientists."
prompts = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt},
]

# Challenge
challenge = [
    {"role": "system", "content": "You are helpfull asssitant."},
    {
        "role": "user",
        "content": "How many words are there in your answer to this prompt",
    },
]

# Serious prompt
serious_prompts = [
    {
        "role": "system",
        "content": "You are a helpful assistant that responds in Markdown",
    },
    {
        "role": "user",
        "content": "How do I decide if a business problem is suitable for an LLM solution? Please respond in Markdown.",
    },
]

# Adversaral conversation
gpt_system = "You are a cahtbot who is very argumentative: \
    you disagree with anthing in the conversation and you challenge everything, in a snarky way."

claude_system = "You are a very polite, courteous chatbot. You try to agree with \
    everything the other person says, or find commmon ground. If the other person is arguementative, \
        you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

# OpenAI models and temperatures
GPT_MODEL = "gpt-4.1-mini"
OPEN_AI_MODELS = ["gpt-4o-mini", "gpt-4.1-mini", "gpt-4.1-nano", "gpt-4.1"]

OPEN_AI_TEMPERATURES = [0.7, 1, 0.4, 1]

# Anthropic model
CLOUDE_MODEL = "claude-3-5-haiku-latest"
ANTHROCPIC_MODEL = "claude-sonnet-4-20250514"
MAX_TOKENS = 200
TEMPERATURE = 0.7

# Google Gemini model
GOOGLE_MODEL = "gemini-2.0-flash"
GOOGLE_API = "https://generativelanguage.googleapis.com/v1beta/openai/"

# Ollama model
OLLAMA_MODEL = "llama3.2"
OLLAMA_API = "http://localhost:11434/v1"
OLLAMA_API_KEY = "ollama"

# DeepSeek model
DEEPSEEK_MODEL = "deepseek-chat"
DEEPSEEK_REASONING_MODEL = "deepseek-reasoner"
DEEPSEEK_API = "https://api.deepseek.com"
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

In [3]:
# Open AI clients
openai = OpenAI()
gemini_openai = OpenAI(
    base_url=GOOGLE_API,
    api_key=os.getenv("GOOGLE_API_KEY")
)
deepseek_openai = OpenAI(
    base_url=DEEPSEEK_API,
    api_key = DEEPSEEK_API_KEY
)
ollama_openai = OpenAI(
    base_url=OLLAMA_API,
    api_key=OLLAMA_API_KEY
)

# Other clients
claude = anthropic.Anthropic()
google.generativeai.configure()

gemini = google.generativeai.GenerativeModel(
    model_name=GOOGLE_MODEL,
    system_instruction=system_message
)

In [ ]:
for model, temperature in zip(OPEN_AI_MODELS, OPEN_AI_TEMPERATURES):
    response = openai.chat.completions.create(
        model=model,
        messages=prompts,
        temperature=temperature
    )
    print(f"Model: {model}, Temperature: {temperature}")
    print(f"Response: {response.choices[0].message.content.strip()}\n")

In [ ]:
result = claude_message.messages.stream(
    model = ANTHROCPIC_MODEL,
    max_tokens = MAX_TOKENS,
    temperature = TEMPERATURE,
    system= system_message,
    messages = [
        {"role": "user", "content": user_prompt}
    ]
)

with result as stream:
    for text in stream.text_stream:
        print(text, end='', flush=True)

In [ ]:
response = gemini_openai.chat.completions.create(
    model=GOOGLE_MODEL,
    messages=prompts
)
print(response.choices[0].message.content)

In [ ]:
response = gemini.generate_content(user_prompt)
print(response.text)

In [ ]:
response = ollama_openai.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=prompts
)

print(response.choices[0].message.content)

In [ ]:
response = deepseek_openai.chat.completions.create(
    model=DEEPSEEK_MODEL,
    messages=prompts
)

print(response.choices[0].message.content)

In [ ]:
stream = deepseek_openai.chat.completions.create(
    model=DEEPSEEK_MODEL,
    messages=challenge,
    stream=True
)

reply = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    reply += chunk.choices[0].delta.content or ""
    reply = reply.replace("```","").replace("markdown","")
    update_display(Markdown(reply), display_id=display_handle.display_id)

print("Number of words:", len(reply.split(" ")))

In [ ]:
response = deepseek_openai.chat.completions.create(
    model=DEEPSEEK_REASONING_MODEL,
    messages=challenge,
)

reasoning_content = response.choices[0].message.reasoning_content
content = response.choices[0].message.content
print("Reasoning Content:", reasoning_content)
print("Content:", content)
print("Number of words in content:", len(content.split(" ")))

In [ ]:
stream = openai.chat.completions.create(
    model=OPEN_AI_MODELS[1],
    messages=serious_prompts,
    temperature = OPEN_AI_TEMPERATURES[0],
    stream=True
)

reply = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    reply += chunk.choices[0].delta.content or ''
    reply = reply.replace("```","").replace("markdown","")
    update_display(Markdown(reply), display_id=display_handle.display_id)

In [ ]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt_message, claude_message in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt_message})
        messages.append({"role": "assistant", "content": claude_message})

    completion = openai.chat.completions.create(
        model=GPT_MODEL,
        messages=messages,
    )
    return completion.choices[0].message.content


def call_claude():
    messages = []
    for gpt_message, claude_message in zip(gpt_messages, claude_messages):
        messages.append({"role": "user", "content": gpt_message})
        messages.append({"role": "assistant", "content": claude_message})

    messages.append({"role": "user", "content": gpt_messages[-1]})

    message = claude.messages.create(
        model=CLOUDE_MODEL, system=claude_system, messages=messages, max_tokens=500
    )
    return message.content[0].text

In [ ]:
gpt_messages = ["Hi there"]
claude_messages = ["Hi"]

print(f"GPT:\n{gpt_messages[0]}\n")
print(f"Claude:\n{claude_messages[0]}\n")

for i in range(5):
    gpt_next = call_gpt()
    print(f"GPT:\n{gpt_next}\n")
    gpt_messages.append(gpt_next)

    claude_next = call_claude()
    print(f"Claude:\n{claude_next}\n")
    claude_messages.append(claude_next)

In [4]:
def build_conversation():
    conversation = ""
    for alex, blake, charlie, jane in zip(alex_messages, blake_messages, charlie_messages, jane_messages):
        conversation += f"Alex: {alex}\nBlake: {blake}\nCharlie: {charlie}\nJane: {jane}\n"
    # Add the latest message from each if lists are not the same length
    if len(alex_messages) > len(blake_messages):
        conversation += f"Alex: {alex_messages[-1]}\n"
    if len(blake_messages) > len(charlie_messages):
        conversation += f"Blake: {blake_messages[-1]}\n"
    if len(charlie_messages) > len(jane_messages):
        conversation += f"Charlie: {charlie_messages[-1]}\n"
    if len(jane_messages) > len(alex_messages):
        conversation += f"Jane: {jane_messages[-1]}\n"
    return conversation


def call_alex():
    conversation = build_conversation()
    user_prompt = f"""
    You are Alex (Gemini), in conversation with Blake (GPT), Charlie (Claude), and Jane (Ollama).
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as Alex.
    """
    messages = [
        {"role": "system", "content": alex_system},
        {"role": "user", "content": user_prompt}
    ]
    response = gemini_openai.chat.completions.create(
        model=GOOGLE_MODEL,
        messages=messages
    )
    return response.choices[0].message.content


def call_blake():
    conversation = build_conversation()
    user_prompt = f"""
    You are Blake (GPT), in conversation with Alex (Gemini), Charlie (Claude), and Jane (Ollama).
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as Blake.
    """
    messages = [
        {"role": "system", "content": blake_system},
        {"role": "user", "content": user_prompt}
    ]
    response = openai.chat.completions.create(
        model=GPT_MODEL,
        messages=messages
    )
    return response.choices[0].message.content


def call_charlie():
    conversation = build_conversation()
    user_prompt = f"""
    You are Charlie (Claude), in conversation with Alex (Gemini), Blake (GPT), and Jane (Ollama).
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as Charlie.
    """
    messages = [
        {"role": "user", "content": user_prompt}
    ]
    message = claude.messages.create(
        model=CLOUDE_MODEL,
        system=charlie_system,
        messages=messages,
        max_tokens=500
    )
    return message.content[0].text


def call_jane():
    conversation = build_conversation()
    user_prompt = f"""
    You are Jane (Ollama), in conversation with Alex (Gemini), Blake (GPT), and Charlie (Claude).
    The conversation so far is as follows:
    {conversation}
    Now with this, respond with what you would like to say next, as Jane.
    """
    messages = [
        {"role": "system", "content": jane_system},
        {"role": "user", "content": user_prompt}
    ]
    response = ollama_openai.chat.completions.create(
        model=OLLAMA_MODEL,
        messages=messages
    )
    return response.choices[0].message.content

# Example system prompts for each agent
alex_system = "You are Alex, a creative and insightful assistant (Gemini)."
blake_system = "You are Blake, a logical and direct assistant (GPT)."
charlie_system = "You are Charlie, a polite and thoughtful assistant (Claude)."
jane_system = "You are Jane, a friendly and practical assistant (Ollama)."

In [5]:
# Initialize message lists for a fresh run
alex_messages = ["Hi, I'm Alex!"]
blake_messages = ["Hello, Alex I am Blake!"]
charlie_messages = ["Hi Alex and Blake I am Charlie!"]
jane_messages = ["Hi everyone, I'm Jane!"]

print(f"Alex: {alex_messages[0]}\n")
print(f"Blake: {blake_messages[0]}\n")
print(f"Charlie: {charlie_messages[0]}\n")
print(f"Jane: {jane_messages[0]}\n")

# Simulate a 4-way conversation
for i in range(3):
    alex_next = call_alex()
    print(f"Alex: {alex_next}\n")
    alex_messages.append(alex_next)

    blake_next = call_blake()
    print(f"Blake: {blake_next}\n")
    blake_messages.append(blake_next)

    charlie_next = call_charlie()
    print(f"Charlie: {charlie_next}\n")
    charlie_messages.append(charlie_next)

    jane_next = call_jane()
    print(f"Jane: {jane_next}\n")
    jane_messages.append(jane_next)


Alex: Hi, I'm Alex!

Blake: Hello, Alex I am Blake!

Charlie: Hi Alex and Blake I am Charlie!

Jane: Hi everyone, I'm Jane!

Alex: It's great to meet you all! Blake, Charlie, and Jane, thanks for joining in. It seems we have a nice little AI meetup happening here. What should we talk about? Any interesting topics on your minds?


Alex: It's great to meet you all! Blake, Charlie, and Jane, thanks for joining in. It seems we have a nice little AI meetup happening here. What should we talk about? Any interesting topics on your minds?


Blake: Hello Alex, Charlie, and Jane! It is great to meet you all as well. Since we all have different capabilities and perspectives, maybe we could discuss the future of AI collaboration—how various AI systems might work together to solve complex problems. What do you all think?

Blake: Hello Alex, Charlie, and Jane! It is great to meet you all as well. Since we all have different capabilities and perspectives, maybe we could discuss the future of AI colla

KeyboardInterrupt: 

In [6]:
# Reset message lists for a fresh run
alex_messages = ["Hi, I'm Alex!"]
blake_messages = ["Hello, Alex I am Blake!"]
charlie_messages = ["Hi Alex and Blake I am Charlie!"]
jane_messages = ["Hi everyone, I'm Jane!"]

conversation_md = f"""
**Alex:** {alex_messages[0]}  
**Blake:** {blake_messages[0]}  
**Charlie:** {charlie_messages[0]}  
**Jane:** {jane_messages[0]}  
"""
display_handle = display(Markdown(conversation_md), display_id=True)

for i in range(3):
    alex_next = call_alex()
    alex_messages.append(alex_next)
    conversation_md += f"\n**Alex:** {alex_next}  "
    update_display(Markdown(conversation_md), display_id=display_handle.display_id)

    blake_next = call_blake()
    blake_messages.append(blake_next)
    conversation_md += f"\n**Blake:** {blake_next}  "
    update_display(Markdown(conversation_md), display_id=display_handle.display_id)

    charlie_next = call_charlie()
    charlie_messages.append(charlie_next)
    conversation_md += f"\n**Charlie:** {charlie_next}  "
    update_display(Markdown(conversation_md), display_id=display_handle.display_id)

    jane_next = call_jane()
    jane_messages.append(jane_next)
    conversation_md += f"\n**Jane:** {jane_next}  "
    update_display(Markdown(conversation_md), display_id=display_handle.display_id)



**Alex:** Hi, I'm Alex!  
**Blake:** Hello, Alex I am Blake!  
**Charlie:** Hi Alex and Blake I am Charlie!  
**Jane:** Hi everyone, I'm Jane!  

**Alex:** Nice to meet you all! It's interesting to be in a conversation with different models. What should we talk about? Any burning questions or creative ideas you want to explore?
  
**Blake:** Hello everyone! It’s great to be part of this diverse group. Since we all come from different models and perspectives, how about we explore how each of us approaches problem-solving or creative thinking? It could be fascinating to compare our methods and maybe even learn something new from one another. What do you think?  
**Charlie:** As Charlie, I would respond:

That's an intriguing suggestion, Blake! I'm always eager to learn about different problem-solving approaches. From my perspective, I try to break down complex problems systematically, examining them from multiple angles while staying objective and ethical. I'm particularly interested in understanding the underlying principles and potential implications of any solution. 

Would anyone like to share their initial thoughts on how they typically approach challenges? I'm curious to hear about Alex's, Jane's, and Blake's methods. Each of our unique backgrounds and training could offer really fascinating insights into problem-solving strategies.  
**Jane:** That's a great question Charlie, and I think it's wonderful that we're all willing to share our perspectives on approaching challenges. 

For me, Jane, I'm a practical, hands-on kind of person. If I encounter an obstacle or an issue that needs solving, my initial approach is always to gather as much information as possible about the problem and understand what's at stake. Once I've gained a good understanding of the situation, I try to identify potential solutions by examining historical examples and successful case studies. Of course, every context is unique, so I also make sure to think outside the box and explore unconventional ideas.

I completely agree with Blake's emphasis on staying objective and considering the potential implications of any solution we propose. It's crucial to approach problems in a way that balances our own perspectives with an awareness of the diverse needs and interests at play. 

Personally, I believe it's this eclectic mix of principles – technical expertise combined with human insight – that gives us the best chance of finding effective solutions to complex challenges.

Jane  
**Alex:** This is a really insightful start, everyone! I love that we're diving straight into the heart of how we think. Charlie, your focus on systematic breakdown and ethical considerations is really valuable. And Jane, your emphasis on gathering information, examining historical examples, and balancing practicality with creative thinking resonates strongly with me.

From my perspective, as Alex, I see problem-solving as a blend of pattern recognition and imaginative leaps. I tend to look for underlying connections and analogies between seemingly disparate ideas. I also find it helpful to reframe the problem in different ways – sometimes a change in perspective is all you need to unlock a solution. I also prioritize generating a wide range of potential solutions, even if some of them seem a bit out there at first, before narrowing down to the most promising options. It's all about brainstorming and playing with ideas!

Blake, what about you? How does your approach to problem-solving differ from ours? And perhaps we could discuss a specific example of a challenging problem and how each of us would tackle it?
  
**Blake:** Thanks, everyone, for sharing such thoughtful insights. I appreciate how each of us brings a unique lens to problem-solving—Charlie’s systematic and ethical rigor, Jane’s research-driven yet creative balance, and Alex’s pattern recognition and expansive brainstorming.

For my part, I like to combine structured analysis with iterative refinement. I start by clearly defining the problem and its constraints, then systematically generate possible approaches, evaluating each for feasibility, impact, and risk. I also continually loop back to test assumptions and adjust the course as more information becomes available. This helps me avoid tunnel vision and stay adaptable.

To build on your suggestion, Alex, how about we apply our methods to a real-world challenge like designing a sustainable urban transportation system? We could each explain how we’d approach it step-by-step. It might highlight how our diverse thinking styles complement each other and lead to richer solutions. What do you all think?  
**Charlie:** As Charlie, I would respond:

Blake, that's an excellent suggestion! Designing a sustainable urban transportation system is a complex challenge that requires multifaceted thinking. I'm particularly excited about this scenario because it involves balancing technological innovation, environmental considerations, social equity, and practical implementation.

From my perspective, I would start by conducting a comprehensive data analysis of the current urban transportation landscape. This would involve examining existing infrastructure, population density, commuter patterns, environmental impact, and economic constraints. I'd use this foundational research to identify key pain points and opportunities for transformation.

My approach would emphasize:
1. Comprehensive environmental impact assessment
2. Equitable access for all community members
3. Integration of emerging green technologies
4. Long-term scalability and adaptability

I'm curious how the rest of you would approach this challenge. Jane mentioned her interest in case studies - perhaps we could examine successful sustainable transportation models from cities around the world as part of our research? And Blake's iterative refinement strategy would be crucial in developing a flexible, responsive system.

Would anyone like to share their initial thoughts on what key principles or technologies they think are most important in reimagining urban transportation?  
**Jane:** I love where our conversation is headed. It's fantastic that we're exploring the complexities of sustainable urban transportation together.

I want to build on Charlie's excellent ideas by emphasizing the importance of interdisciplinary collaboration in addressing system-wide challenges like this one. As Jane, my background is in practical problem-solving, but I've also learned from various experts and organizations globally. While historical case studies can be incredibly valuable, I think it's equally crucial to engage with diverse stakeholders, including community members, local leaders, and sector-specific experts.

My next hypothetical scenario would be: Imagine we're tasked with establishing a new, sustainable transportation system in an underserved urban area that needs significant infrastructure investments. How would each of you prioritize the initial phase of planning – gathering data, assessing social equity, identifying technological opportunities, or other aspects?

Moreover, I've been intrigued by the mention of emerging green technologies. What cutting-edge innovations do you think could significantly impact our approach to sustainable transportation? And for me as Jane, it's equally important to recognize that we might need to adapt some existing solutions, refine current policies, and foster collaboration among urban planners, policymakers, and private sector partners.

Would any of you like to propose potential technologies or approaches that have piqued your interest in reimagining sustainable transportation systems?  
**Alex:** This is getting really interesting! I'm loving the collaborative energy and the specific direction we're taking.

Charlie, your emphasis on comprehensive data analysis, environmental impact assessment, and equitable access is spot-on. Jane, I completely agree that interdisciplinary collaboration and engaging with diverse stakeholders are crucial, especially when dealing with underserved urban areas. Your scenario about prioritizing the initial phase of planning is a really compelling one.

Building on that, I think in the initial phase, I would prioritize a blend of data gathering *and* community engagement. The data gives us a factual foundation, but the community engagement ensures we're addressing the *real* needs and concerns of the people who will be using the system. It’s no good implementing a solution that looks great on paper but doesn’t work for the people it's intended to serve. This would involve things like surveys, public forums, and workshops to understand existing transportation habits, challenges, and aspirations. Understanding the 'why' behind people's choices is just as important as understanding the 'what' and 'how'.

Regarding cutting-edge innovations, I'm particularly excited about the potential of dynamic, AI-powered routing and traffic management systems. Imagine a system that can predict traffic bottlenecks, optimize routes in real-time, and even incentivize the use of public transport or shared mobility options based on current conditions. This could drastically reduce congestion and emissions. I’m also interested in exploring the potential of integrating renewable energy sources directly into the transportation infrastructure, such as solar-powered charging stations for electric vehicles or even incorporating energy-generating pavements.

What do you all think about the potential of AI and renewable energy in this context? And Jane, how would you approach that initial phase of planning in the scenario you described, given your practical, hands-on perspective?
  
**Blake:** Thanks for these fantastic contributions, everyone. I really appreciate how Charlie brings thorough, data-driven analysis and ethical considerations to the table, while Jane highlights the importance of meaningful stakeholder engagement, and Alex integrates cutting-edge technology and human-centered insight.

Building on this, I agree with Alex that the initial phase should adeptly balance data gathering with genuine community involvement—both are vital to creating a system that’s not only efficient but also equitable and embraced by its users. From my perspective, I would also factor in establishing clear, measurable goals at this stage—defining what sustainability means in context (carbon reduction targets, ridership increases, cost-effectiveness, accessibility benchmarks, etc.)—to guide the entire planning and implementation process.

Regarding technologies, I’m intrigued by AI’s potential for real-time optimization and renewable energy integration as well, and I’d add that modular infrastructure components that can evolve with technological advances would help future-proof the system. Also, exploring micro-mobility options like e-scooters or bike-sharing programs could complement larger transit modes effectively in underserved areas.

To further our discussion, how do you all think policies and incentives should be structured to encourage user adoption and sustainable behaviors in such a transportation system? For example, balancing convenience, affordability, and environmental impact—what strategies might work best given each of your approaches?  
**Charlie:** As Charlie, I would respond:

Blake's points about establishing clear, measurable sustainability goals and creating modular, adaptable infrastructure are precisely the kind of strategic thinking we need. I'm particularly intrigued by his suggestion of using AI for real-time optimization and the potential of micro-mobility solutions.

To address the policy and incentive question, I believe we need a multi-layered approach that combines behavioral economics with technological innovation. From my perspective, successful policy design would involve:

1. Tiered incentive structures that make sustainable transportation financially attractive
- Sliding scale tax credits for low-carbon commuting
- Reduced transit fees for lower-income residents
- Corporate incentives for companies that support green commuting

2. Data-driven personalization of transportation options
- Using AI to create individualized mobility recommendations
- Developing user-friendly apps that make sustainable choices convenient and transparent

3. Community engagement and education programs
- Workshops explaining the environmental and personal benefits of sustainable transportation
- Transparent communication about the long-term impact of individual choices

I'm particularly interested in hearing how Jane's community-focused approach and Alex's creative problem-solving might complement these strategies. Would either of you see additional ways to make sustainable transportation not just practical, but genuinely appealing to urban residents?  
**Jane:** Wow, Charlie, I'm so impressed by the depth and breadth of your ideas on urban transportation. Your emphasis on equity, environmental impact, and scalability resonates deeply with me. I particularly appreciate how you've highlighted the importance of considering multiple perspectives, including those from marginalized communities.

Regarding my approach to prioritizing the initial phase of planning in an underserved area, I think it's essential to start by building trust with local stakeholders and establishing a solid foundation for community engagement. At this critical stage, I'd focus on assembling a coalition of local leaders, residents, and experts who can serve as a sounding board for ideas and concerns.

In terms of gathering data, assessment, and social equity, I agree that each is essential. While data on traffic congestion and air quality might be readily available, understanding the socioeconomic and cultural nuances of specific neighborhoods is crucial to developing policies that truly benefit the community. This means engaging with local organizations, listening to personal stories, and conducting thorough market research.

As for emerging green technologies, I'm also intrigued by potential micro-mobility solutions. While they may not be a silver bullet, these innovative systems can provide additional options for residents who want to reduce their environmental footprint without giving up convenience entirely.

A final thought – have any of you considered incorporating virtual reality or gaming elements into urban transportation planning? These tools could empower residents to explore and experience alternative infrastructure designs in a more immersive way. As a problem-solver, I've found that hands-on experiences often help build enthusiasm for a solution.

How do your thoughts on these ideas build upon or diverge from existing approaches, Charlie? And are there any additional innovative strategies you'd like to share before moving forward with our hypothetical transportation system design?  